<a href="https://colab.research.google.com/github/malcolmchanhaoxian/Qwen3ASR-OV-Quant/blob/main/Qwen3ASR_Quant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from pathlib import Path
import requests

#pip install "optimum-intel[openvino]==1.26.1" huggingface_hub[hf_xet] "nncf==2.19.0" "openvino==2025.4.0" "openvino-genai==2025.4.0"
# Fetch notebook_utils module
if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

if not Path("qwen_3_asr_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/qwen3-asr/qwen_3_asr_helper.py",
    )
    open("qwen_3_asr_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-asr.ipynb")

In [6]:
import sys
from cmd_helper import clone_repo
from pip_helper import pip_install
import platform

# Re-run the uninstall to ensure a clean slate
!{sys.executable} -m pip uninstall -y -q torch torchaudio transformers qwen-asr torchvision

In [7]:
# Re-running the main pip install command with verbose output and corrected arguments
!{sys.executable} -m pip install -v \
    --extra-index-url https://download.pytorch.org/whl/cpu \
    torch==2.8.0 \
    torchvision \
    nncf \
    torchaudio==2.8.0 \
    "openvino>=2025.4.0" \
    "gradio>=4.0" \
    huggingface_hub \
    scipy

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
  Obtaining dependency information for torch==2.8.0 from https://download-r2.pytorch.org/whl/cpu/torch-2.8.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl.metadata
  Using cached https://download-r2.pytorch.org/whl/cpu/torch-2.8.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Obtaining dependency information for torchvision from https://download-r2.pytorch.org/whl/cpu/torchvision-0.26.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for nncf from https://files.pythonhosted.org/packages/08/a4/d4586fa0fc2f50e22e0ea6673a43d17e756902c4b5d53bed668024d310f6/nncf-3.1.0-py3-none-any.whl.metadata
  Using cached nncf-3.1.0-py3-none-any.whl.metadata (9.9 kB)
  Obtaining dependency information for torchaudio==2.8.0 from https://download-r2.pytorch.org/whl/cpu/torchaudio-2.8.0%2Bcpu-c

In [8]:
# Clone Qwen3-ASR repository for model code
repo_dir = Path("Qwen3-ASR")
revision = "c17a131fe028b2e428b6e80a33d30bb4fa57b8df"
clone_repo("https://github.com/QwenLM/Qwen3-ASR.git", revision)

pip_install(
    "-e", # Removed -q flag
    str(repo_dir),
)

if platform.system() == "Darwin":
    pip_install("numpy<2.0")

In [11]:
from qwen_3_asr_helper import convert_qwen3_asr_model

from nncf import CompressWeightsMode

model_id = "Qwen/Qwen3-ASR-1.7B"
model_name = model_id.split("/")[-1]
ov_model_dir = Path(f"{model_name}-OV-8bit")

# Convert model to OpenVINO format
# This will skip conversion if the model already exists
convert_qwen3_asr_model(
    model_id=model_id,
    output_dir=ov_model_dir,
    quantization_config={"mode": CompressWeightsMode.INT8_SYM},  # Set to {"mode": CompressWeightsMode.INT8_SYM} for INT8 quantization
)

⌛ Qwen/Qwen3-ASR-1.7B conversion started. Be patient, it may takes some time.
⌛ Load Original model


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

✅ Thinker audio model (Conv2D part) successfully converted
⌛ Convert thinker audio encoder model (Transformer layers)
✅ Thinker audio encoder model (Transformer layers) successfully converted
⌛ Convert Thinker Language model


/usr/local/lib/python3.12/dist-packages/transformers/cache_utils.py:132: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not self.is_initialized or self.keys.numel() == 0:
/content/Qwen3-ASR/qwen_asr/core/transformers_backend/modeling_qwen3_asr.py:1021: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if position_ids.ndim == 3 and position_ids.shape[0] == 4:
/content/qwen_3_asr_helper.py:119: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be 

✅ Thinker language model successfully converted
⌛ Weights compression with int8_sym mode started
INFO:nncf:Statistics of the bitwidth distribution:
┍━━━━━━━━━━━━━━━━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┑
│ Weight compression mode   │ % all parameters (layers)   │ % ratio-defining parameters (layers)   │
┝━━━━━━━━━━━━━━━━━━━━━━━━━━━┿━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┿━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┥
│ int8_sym, per-channel     │ 100% (197 / 197)            │ 100% (197 / 197)                       │
┕━━━━━━━━━━━━━━━━━━━━━━━━━━━┷━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┷━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┙


Output()

✅ Weights compression finished
✅ Thinker model conversion finished. You can find results in Qwen3-ASR-1.7B-OV-8bit
✅ Qwen/Qwen3-ASR-1.7B model conversion finished. You can find results in Qwen3-ASR-1.7B-OV-8bit
